In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Stichproben-baseerte Quantendiagonalisierung vun een Chemie-Hamiltonian

*Bruukschattung: ünner een Minuut op een Heron r2 Prozessor (HENWIES: Dat is blots een Schattung. Dien Looptiet kann variern.)*
## Achtergrund
In dit Tutorial wiesen wi, wo verrüschte Quantenstichproben naobearbeidt warrt, um een Approximation vun den Grundtostand vun dat Stickstoffmolekül $\text{N}_2$ bi Gliekgewichtsbindungslängde to finnen. Dobi bruukt wi den [stichproben-baseerten Quantendiagonalisierungsalgorithmus (SQD)](https://arxiv.org/abs/2405.05068) för Stichproben ut een 59-Qubit-Quantenschaltkreis (52 System-Qubits + 7 Ancilla-Qubits). Een Qiskit-baseerte Implementierung is verföögbar in de [SQD Qiskit Addons](https://github.com/Qiskit/qiskit-addon-sqd), mehr Details finnen Se in de entsprechende [Dokumentation](/guides/qiskit-addons-sqd) mit een [eenfach Bispeel](/guides/qiskit-addons-sqd-get-started) to't Anfangen.

SQD is een Technik to't Finnen vun Eigenwerten un Eigenvektoren vun Quantenoperatoren, as den Hamiltonian vun een Quantensystem, ünner Bruuk vun Quanten- un verdeelte klassische Computing tosamen. Dat verdeelte klassische Computing warrt bruukt, um Stichproben vun een Quantenprozessor to verarbeiten un een Teel-Hamiltonian in een dör ehr opspannte Ünnerruum to projezeren un to diagonaliseren. Dat maakt SQD robust gegen Stichproben, de dör Quantenrüschen verfälscht sünd, un to't Ümgahn mit grote Hamiltonians, as Chemie-Hamiltonians mit Millionen vun Wechselwirkungstermen, buten de Riekwiedte vun exakte Diagonaliseringsmethoden. Een SQD-baseerte Workflow hett folgend Schritten:

1. Wähl een Schaltkreis-Ansatz un wend em op een Quantencomputer op een Referenztostand an (in dit Fall den [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)-Tostand).
2. Sample Bitstrings ut den resulterende Quantentostand.
3. Föhr dat *sülvkonsistente Konfigurationswedderherstellenverfahren* op de Bitstrings ut, um de Approximation vun den Grundtostand to kriegen.

SQD is bekannt dorför, goot to arbeiten, wenn de Teel-Eigentostand dünn besett is: De Wellenfunktion warrt dregen vun een Menge vun Basistoständ $\mathcal{S} = {|x\rangle }$, ehr Grött nich exponentiell mit de Problemgrött wussen deit.

### Quantenchemie
De Eigenschaften vun Molekülen warrt grootdeels dör de Struktur vun de Elektronen in ehr bestimmt. As fermionische Deelken künnt Elektronen mit een mathematischen Formalismus, Tweedquantisierung nömmt, beschreven warrn. De Idee is, dat dat een Antall vun *Orbitalen* gifft, vun de jeedeen entweder leer is or vun een Fermion besett warrt. Een System vun $N$ Orbitalen warrt dör een Satz fermionischer Vernichtungsoperatoren ${\hat{a}_p}_{p=1}^N$ beschreven, de de fermionischen Antikommutatorrelatschonen erfüllen:

$$
\begin{align*}
\hat{a}_p \hat{a}_q + \hat{a}_q \hat{a}_p &= 0, \\
\hat{a}_p \hat{a}_q^\dagger + \hat{a}_q^\dagger \hat{a}_p &= \delta_{pq}.
\end{align*}
$$

De adjungeerde Operator $\hat{a}_p^\dagger$ warrt Erzeugungsoperator nömmt.

Bit nu hett uns Darstellen de Spin nich berücksichtigt, de een fundamentale Eigenschaft vun Fermionen is. Bi't Berücksichtigen vun Spin kamen de Orbitalen in Poren vör, *Ruumorbitalen* nömmt. Jeed Ruumorbital besteiht ut twee *Spinorbitalen*, vun de een mit "Spin-$\alpha$" un een mit "Spin-$\beta$" tekent warrt. Wi schrieven denn $\hat{a}_{p\sigma}$ för den Vernichtungsoperator, de mit dat Spinorbital mit Spin $\sigma$ ($\sigma \in {\alpha, \beta}$) in Ruumorbital $p$ verbunnen is. Wenn wi $N$ as Antall vun de Ruumorbitalen nehmen, gifft dat insgesamt $2N$ Spinorbitalen. De Hilbert-Ruum vun dit System warrt vun $2^{2N}$ orthonormale Basisvektoren opspannt, de mit tweedelige Bitstrings tekent warrn: $\lvert z \rangle = \lvert z_\beta z_\alpha \rangle = \lvert z_{\beta, N} \cdots z_{\beta, 1} z_{\alpha, N} \cdots z_{\alpha, 1} \rangle$.

De Hamiltonian vun een molekular System kann schreven warrn as

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

wobei de $h_{pr}$ un $h_{prqs}$ komplexe Tahlen sünd, de molekulare Integrale nömmt warrn un ut de Spezifikation vun dat Molekül mit een Computerprogramm berekent warrn künnt. In dit Tutorial berekent wi de Integrale mit dat Softwarepaket [PySCF](https://pyscf.org/).

För Details dorüm, wo de molekulare Hamiltonian herleidt warrt, konsulteer een Lehrbök to Quantenchemie (to'n Bispeel *Modern Quantum Chemistry* vun Szabo un Ostlund). För een högerordnete Verklaren, wo Quantenchemieprobleme op Quantencomputer afbildt warrn, kiek di de Vorlesung [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE&t=900) vun de Qiskit Global Summer School 2024 an.

### Local Unitary Cluster Jastrow (LUCJ) Ansatz
SQD bruukt een Quantenschaltkreis-Ansatz, ut den Stichproben trocken warrn künnt. In dit Tutorial bruukt wi den [Local Unitary Cluster Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) Ansatz wegen sien Kombination ut physikalische Motivation un Hardware-Fründlichkeit.

De LUCJ-Ansatz is een spezialiseerde Form vun den allgemenen Unitary Cluster Jastrow (UCJ) Ansatz, de de Form hett

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

wobei $\lvert \Phi_0 \rangle$ een Referenztostand is, faken de Hartree-Fock-Tostand, un de $\hat{K}_\mu$ un $\hat{J}_\mu$ de Form hebbt

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

wobei wi den Deelkentall-Operator defineert hebbt

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

De Operator $e^{\hat{K}_\mu}$ is een Orbitalrotation, de mit bekannte Algorithmen in linearer Deepte un ünner Bruuk vun linearer Konnektivität implementeert warrn kann.

De Implementierung vun den $e^{i \mathcal{J}_k}$ Term vun den UCJ-Ansatz fordert entweder vollständige Konnektivität or den Bruuk vun een fermionisch Swap-Nettwark, wat för verrüschte prä-fehlertolerante Quantenprozessoren mit begrenzte Konnektivität een Roperfordderung is. De Idee vun den *lokalen* UCJ-Ansatz is, Dünnbessettheitsbedingen op de $\mathbf{J}^{\alpha\alpha}$- un $\mathbf{J}^{\alpha\beta}$-Matrizen opto leggen, de dat möglich maken, ehr in konstante Deepte op Qubit-Topologien mit begrenzte Konnektivität to implementeren. De Bedingen warrn dör een List vun Indizes spezifizieret, de anwiesen, welk Matrixindrääg in dat bövere Drieheck vun null verscheden sien dröfft (do de Matrizen symmetrisch sünd, mutt blots dat bövere Drieheck spezifizieret warrn). Disse Indizes künnt as Orbitalpaore interpratseert warrn, de mitanner interageren dröfft.

Betrach as Bispeel een quadratisch Gitter-Qubit-Topologie. Wi künnt de $\alpha$- un $\beta$-Orbitalen in parallele Lienen op dat Gitter platzeren, wobei Verbinnen twischen de Lienen "Sprossen" vun een Leiterforrm bilden, as folgt:

![Qubit-Toordenungsdiagramm för den LUCJ-Ansatz op een quadratisch Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/baad4e53-5bfd-4cb4-8027-2ec26d27ecdd.avif)

Bi dit Setup sünd Orbitalen mit den sülven Spin mit een Linientopologie verbunnen, wiel Orbitalen mit ünnersch Spin verbunnen sünd, wenn se dat sülve Ruumorbital delen. Dat gifft de folgend Indexbedingen för de $\mathbf{J}$-Matrizen:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, \ldots, N-1}
\end{align*}
$$

Mit anner Wöör: Wenn de $\mathbf{J}$-Matrizen blots an de angeven Indizes in dat bövere Drieheck vun null verscheden sünd, kann de $e^{i \mathcal{J}_k}$ Term op een quadratische Topologie ahn Bruuk vun Swap-Gates in konstante Deepte implementeert warrn. Natüürlich maakt dat Opleggen vun söke Bedingen op den Ansatz em weniger utdrucksstark, sodass villicht mehr Ansatz-Wedderholen nöödig sünd.

De IBM&reg; Hardware hett een Heavy-Hex-Gitter-Qubit-Topologie, in welk Fall wi een "Zickzack"-Muster bruken künnt, dat ünnen darstallt is. In dit Muster warrn Orbitalen mit den sülven Spin op Qubits mit een Linientopologie afbildt (roode un blaue Kringe), un een Verbinnen twischen Orbitalen vun ünnersch Spins is an jeeden 4. Ruumorbital vörhannen, wobei de Verbinnen dör een Ancilla-Qubit (violette Kringe) möglich maakt warrt. In dit Fall sünd de Indexbedingen

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, 4, 8, \ldots (p \leq N-1)}
\end{align*}
$$

![Qubit-Toordenungsdiagramm för den LUCJ-Ansatz op een Heavy-Hex-Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Sülvkonsistente Konfigurationswedderherstellung
Dat sülvkonsistente Konfigurationswedderherstellenverfahren is dorop utleegt, so veel Signal as möglich ut verrüschte Quantenstichproben ruttohalen. Do de molekulare Hamiltonian de Deelkentall un Spin-Z erholen deit, maakt dat Sinn, een Schaltkreis-Ansatz to wählen, de disse Symmetrien ok erholen deit. Wenn he op den Hartree-Fock-Tostand anwendt warrt, hett de resulterende Tostand in den rüschenfrijen Fall een faste Deelkentall un Spin-Z. Dorher schullen de Spin-$\alpha$- un Spin-$\beta$-Hälvten vun jeeden ut den Tostand sampelte Bitstring dat sülve [Hamming-Gewicht](https://en.wikipedia.org/wiki/Hamming_weight) as in den Hartree-Fock-Tostand hebben. Wegen dat Vörhannen sien vun Rüschen in aktuelle Quantenprozessoren warrn welk mäten Bitstrings disse Eigenschaft verletzen. Een eenfache Forrm vun Postselektion würr disse Bitstrings wegsmiten, aver dat is verspillerisch, wiel disse Bitstrings villicht noch een beten Signal enthalden. Dat sülvkonsistente Wedderherstellenverfahren versöcht, een Deel vun dat Signal in de Naobearbeiten wedder hertostellen. Dat Verfahren is iterativ un bruukt as Ingaav een Schattung vun de dörchsnittlichen Bessetten vun jeeden Orbital in den Grundtostand, de toeerst ut de Rohstichproben berekent warrt. Dat Verfahren warrt in een Slööp utföhrt, un jeed Iteration hett de folgend Schritten:

1. För jeeden Bitstring, de de spezifizierten Symmetrien verletzt, flipp sien Bits mit een probabilistisch Verfahren, dat dorop utleegt is, den Bitstring neger an de aktuelle Schattung vun de dörchsnittlichen Orbitalbessetten to bringen, um een nijen Bitstring to kriegen.
2. Sammel all olen un nijen Bitstrings, de de Symmetrien erfüllen, un trock Deelmengen vun een in't Vörus wählte faste Grött.
3. För jeed Deelmenge vun Bitstrings projezeer den Hamiltonian in den dör de entsprechend Basisvektoren opspannte Ünnerruum (kiek in den [vörherigen Afsnitt](#quantum-chemistry) för een Beschrieven vun de Basisvektoren) un berekne een Grundtostandsschattung vun den projezeerten Hamiltonian op een klassischen Computer.
4. Aktualiseer de Schattung vun de dörchsnittlichen Orbitalbessetten mit de Grundtostandsschattung mit de neddrigste Energie.

### SQD-Workflow-Diagramm
De SQD-Workflow is in dat folgend Diagramm darstallt:

![Workflow-Diagramm vun den SQD-Algorithmus](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Anfordderungen
Stell seker vör't Anfangen vun dit Tutorial, dat du Folgendes installeert hest:

- Qiskit SDK v1.0 or höger, mit [Visualisierens](https://docs.quantum.ibm.com/api/qiskit/visualization)-Ünnerst Süttung
- Qiskit Runtime v0.22 or höger (`pip install qiskit-ibm-runtime`)
- SQD Qiskit Addon v0.11 or höger (`pip install qiskit-addon-sqd`)
- ffsim v0.0.58 or höger (`pip install ffsim`)
## Setup

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Schritt 1: Klassische Ingaav op een Quantenproblem afbilden
In dit Tutorial finnen wi een Approximation vun den Grundtostand vun dat Molekül in't Gliekgewicht in den cc-pVDZ-Basisatz. Toeerst spezifizeren wi dat Molekül un sien Eigenschaften.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Vör't Konstrueren vun den LUCJ-Ansatz-Schaltkreis föhrt wi toeerst een CCSD-Bereknung in de folgend Code-Zell dör. De [$t_1$- un $t_2$-Amplituden](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) ut de Bereknung warrn bruukt, um de Parameters vun den Ansatz to initialiseren.

In [ ]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]

# Let generate_lucj_pass_manager determine the alpha-beta interactions
pairs_ab = None

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell
    # from running too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it
# to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Nu bruukt wi [ffsim](https://github.com/qiskit-community/ffsim), um den Ansatz-Schaltkreis to maken. Do uns Molekül een slaten-schalich Hartree-Fock-Tostand hett, bruukt wi de spin-balanzierte Variante vun den UCJ-Ansatz, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Wi geven Wechselwirkungspaore, de för een Heavy-Hex-Gitter-Qubit-Topologie passt (kiek in den [Achtergrundafsnitt to'n LUCJ-Ansatz](#local-unitary-cluster-jastrow-lucj-ansatz)). Wi setten `optimize=True` in de `from_t_amplitudes`-Methode, um de "komprimierte" Doppelfaktorisierung vun de $t_2$-Amplituden to möglich maken (kiek [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) in de ffsim-Dokumentation för Details).

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


Wi empfehlen de folgend Schritten, um den Ansatz to optimeren un hardware-kompatibel to maken.

- Wähl physikalische Qubits (`initial_layout`) ut de Teel-Hardware ut, de to dat vörher beschreven "Zickzack"-Muster passt. Dat Anleggen vun Qubits in dit Muster föhrt to een effizienten hardware-kompatibeln Schaltkreis mit weniger Gates. Hier fögen wi Code in, um de Utwahl vun dat "Zickzack"-Muster to automatiseren, de een Bewertensh euristik bruukt, um de mit dat utwählte Layout verbunnen Fehler to minimeren.
- Genereer een stufte Pass-Manager mit de Funktion [generate_preset_pass_manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) vun Qiskit mit dien Wahl vun `backend` un `initial_layout`.
- Sett de `pre_init`-Stuuv vun dien stufte Pass-Manager op `ffsim.qiskit.PRE_INIT`. `ffsim.qiskit.PRE_INIT` enthöllt Qiskit-Transpiler-Passes, de Gates in Orbitalrotationen updeelt un denn de Orbitalrotationen tosamanföhrt, wat to weniger Gates in den enngültigen Schaltkreis föhrt.
- Föhr den Pass-Manager op dien Schaltkreis ut.
<details>
<summary>Code för automatiseerde Utwahl vun dat "Zickzack"-Layout</summary>

</details>

In [ ]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: "
    f"{expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

## Schritt 3: Utföhren mit Qiskit-Primitiven
Na de Optimierung vun den Schaltkreis för de Hardware-Utföhren sünd wi praat, em op de Teel-Hardware uttof öhren un Stichproben för de Grundtostandsenergieschattung to sammeln. Do wi blots een Schaltkreis hebbt, bruukt wi den [Job-Utföhrenmodus](/guides/execution-modes) vun Qiskit Runtime un föhrt uns Schaltkreis ut.

In [ ]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]

# Let generate_lucj_pass_manager determine the alpha-beta interactions
pairs_ab = None

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell
    # from running too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it
# to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: "
    f"{expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the
# orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the
# sci_solver argument in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>